# 03. Exploratory Data Analysis: Food Nutrition Databases & Harmonization

**Datasets:** USDA Comprehensive, Healthy Foods Database, Open Food Facts  
**Unified Output:** `data/processed/foods.parquet` (32,847 records)

---
### Analysis Goals:
1. Macro and calorie distributions per 100g.
2. Missingness before and after category-median imputation.
3. Health Score v2 distribution across categories.
4. Allergen prevalence breakdown.


In [ ]:
import pandas as pd

df = pd.read_parquet("../data/processed/foods.parquet")
print(f"Harmonized dataset shape: {df.shape}")
print(df[["name", "source", "category", "kcal_100g", "protein_100g", "health_score_v2"]].head(5))

### 1. Macronutrient Summaries per 100g

In [ ]:
macro_cols = [
    "kcal_100g",
    "protein_100g",
    "fat_100g",
    "carbs_100g",
    "fiber_100g",
    "sugar_100g",
    "sodium_mg_100g",
]
df[macro_cols].describe().round(2)

### 2. Category Distribution & Health Score v2 Ranking

In [ ]:
cat_stats = (
    df.groupby("category")
    .agg(
        count=("food_id", "count"),
        mean_kcal=("kcal_100g", "mean"),
        mean_protein=("protein_100g", "mean"),
        mean_fiber=("fiber_100g", "mean"),
        mean_health_score=("health_score_v2", "mean"),
    )
    .sort_values(by="mean_health_score", ascending=False)
)
cat_stats.round(1)

### 3. Allergen Prevalence across Catalog

In [ ]:
allergen_cols = [c for c in df.columns if c.startswith("allergen_") and c != "allergen_flags"]
allergen_summary = {}
for c in allergen_cols:
    allergen_summary[c.replace("allergen_", "")] = df[c].value_counts().to_dict()
pd.DataFrame(allergen_summary).fillna(0).astype(int)